<a href="https://colab.research.google.com/github/DaviOSad/California-Housing-Clustering/blob/main/TP_de_ICD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Entrega 2: Trabalho Prático de ICD
# Mercado Imobiliário da Califórnia com Base no Censo de 1990

#### Integrantes:
- Davi Sad (2023096043)
- Giordano Liporati (2023027963)
- João Gabriel Vaz (2023099166)
- Rafael Sant'ana (2023087834)

## Análise Exploratória de Dados

In [ ]:
from pathlib import Path
import urllib.request
import pandas as pd
import numpy as np

# Cria uma cópia local do conjunto de dados em /content/dados/housing.csv
# (caso ainda não exista) e retorna um DataFrame com esses dados
def carregar_conjunto_de_dados():
    path_do_csv = Path("dados/housing.csv")

    if not path_do_csv.is_file():
        Path("dados").mkdir(parents=True, exist_ok=True)
        url_dos_dados = "https://raw.githubusercontent.com/ageron/data/main/housing/housing.csv"
        urllib.request.urlretrieve(url_dos_dados, path_do_csv)

    return pd.read_csv(path_do_csv)

# DataFrame sob o qual a análise vai ser realizada
df = carregar_conjunto_de_dados()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB


In [ ]:
import plotly.express as px

fig = px.scatter(df, x="population", y="households", color="ocean_proximity")
fig.show()

### 1. Qual a relação entre a renda média dos moradores da região e o preço do imóvel?


In [ ]:
# Parece que as medianas são artificialmente limitadas
# a 500 mil doláres por residência.
df["median_house_value"].max()

500001.0

In [ ]:
fig = px.scatter(df, x="median_income", y="median_house_value", color="ocean_proximity")
fig.show()

Eixo x:
> **median_income:** renda mediana das famílias residentes (em dezenas de milhares de dólares).

Eixo y:
> **median_house_value:** valor mediano dos imóveis no grupo de blocos (em dólares).

Os preços dos imóveis paracem aumentar conforme a renda mediana dos moradores aumenta, até um limite máximo de 500k doláres.
Podemos ver que a distância ao mar também influencia nessa correlação.

### 2. Existe correlação entre preço e distância do mar? Se sim, positiva ou negativa?

In [ ]:
fig = px.violin(df, x="ocean_proximity", y="median_house_value", box=True, points="all",
                labels={"ocean_proximity": "Proximidade do Oceano",
                        "median_house_value": "Valor Mediano das Casas"},
                color="ocean_proximity",
                category_orders={
                 "ocean_proximity": [
                     "INLAND", "<1H OCEAN", "NEAR OCEAN", "NEAR BAY", "ISLAND"]})
fig.show()

O gráfico acima relaciona a distância das casas ao oceano e seu preço, evidenciando a densidade de casas com determinado valor X para cada medida de distância. A partir dele pode-se derivar algumas hipóteses:

1.   As casas que ficam distantes ao oceano (INLAND) tendem a ter um preço baixo, com grande concentração em preços entre 100 mil e 200 mil dólares. Além disso, há poucas casas com valores acima de 300 mil dólares.
2.  Casas em ilhas (ISLAND) tendem a ter um preço elevado, iniciando em aproximadamente 300 mil dólares. Porém, essas casas são poucas.
3. É possível perceber um aumento gradual dos preços à medida que as casas se aproximam do oceano. Enquanto as casas INLAND tem seu preço concentrado em torno de 100 mil dólares, casas próximas ao oceano tem preços mais bem distribuídos, com valores mais concentrados entre 150 e 300 mil dólares. Ainda, casas não tão próximas mas a menos de 1 hora do oceano tendem a ter preços entre 150 e 200 mil dólares.
4. O preço das casas mostra ser mais bem distribuído próximo ao oceano, enquanto dentro do continente a concentração de preços baixo é notável.





### 3. O quanto a idade do imóvel afeta no preço?

In [ ]:
df_violin = df.copy()
bins = [0, 10, 20, 30, 40, 50, 60]
labels = ['0-10', '11-20', '21-30', '31-40', '41-50', '51-60']
df_violin['age_group'] = pd.cut(df_violin['housing_median_age'], bins=bins, labels=labels, right=True)

fig = px.violin(df_violin, x='age_group', y='median_house_value',box=True, points='all',
    title='Distribuição do Valor dos Imóveis por Faixa Etária',
    labels={
        'age_group': 'Faixa de Idade do Imóvel',
        'median_house_value': 'Valor Mediano do Imóvel'},
        category_orders={
        'age_group': ['0-10', '11-20', '21-30', '31-40', '41-50', '51-60']
    }, color='age_group')

fig.show()

O gráfico mostra a correlação entre a idade de um imóvel e seu preço. Dele, pode-se tirar as seguintes hipóteses:


1.   Imóveis mais novos tendem a ter preços mais baixos, na faixa de 100 a 200 mil dólares. Isso pode ser uma tentativa de construir casas mais acessíveis para a população sempre crescente.
2.   Quanto mais antigos os imóveis, mais bem distribuídos são os preços.
3.   A idade do imóvel não aparenta ter impacto significativo no valor, mas análises mais profundas serão necessárias.

### 4. Seriam a latitude e a longitude boas métricas para estimar o custo do imóvel?

In [ ]:
fig = px.scatter_geo(df,lat='latitude',lon='longitude', color='median_house_value', color_continuous_scale='Viridis',
    title='Preço dos Imóveis por Localização Geográfica', labels={'median_house_value': 'Valor Mediano das Casas'}, scope='usa')
fig.show()


In [ ]:
fig = px.scatter(df, x='longitude', y='latitude', color='median_house_value', color_continuous_scale='Viridis',
    title='Localização dos Imóveis e seus Valores',labels={
        'longitude': 'Longitude',
        'latitude': 'Latitude',
        'median_house_value': 'Valor Mediano'})
fig.update_layout(yaxis=dict(scaleanchor="x", scaleratio=1))  # Mantém escala geográfica
fig.show()


Ambos os gráfico mostram a relação entre a latitude e longitude, e o preço dos imóveis. Conclusões:

1.  O impacto entre as medidas de latitude e longitude no preço não pode ser percebido a partir do gráfico.
2.  Os gráficos reforçam a hipótese de que a proximidade com o oceado siginifica um aumento no preço dos imóveis, enquanto o preço permanece muito baixo para dentro do continente



## Testes de Hipótese e Intervalos de Confiança

#### Nota importante sobre a metodologia

Aqui, é importante pontuar que vamos usar da função $pearsonr$, do scipy. Ela retorna uma tupla (Correlação, pvalor), onde o pvalor é baseado no caso em que a hipótese nula seja "Não há correção linear entre X e Y".

### 1. Qual a relação entre a renda média dos moradores da região e o preço do imóvel?
  
  $H_0$: Não há relação entre a renda média dos moradores da
  região e o preço do imóvel.

  $H_A$: Há relação entre a renda média dos moradores da região e o preço do imóvel.
  Forma de teste: Teste de correlação de Pearson.

In [ ]:
from scipy.stats import pearsonr
import sklearn

corr, pval = pearsonr(df['median_income'], df['median_house_value']) # resposta pra 1?

print(f"correlação é {corr}")
print(f"pvalor é {pval:.4f}")

correlação é 0.688075207958548
pvalor é 0.0000


A intuição aqui é que, caso não houvesse correlação, teriamos um valor próximo de $0$ no pearson. No entanto, temos um valor próximo a $69\%$, o que indica que provavelmente temos uma correlação. Então, precisamos avaliar qual a $P(\text{Observarmos uma correlação de 69% |} H_0 \text{ é verdadeiro})$. A probabilidade dá quase $0\%$, o que usando o nivel de significancia de $5\%$, nos indica que rejeitamos a $H_0$ em favor da $H_A$.



Aqui podemos usar o coeficiente de Pearson porque estamos procurando uma tendência no dado de "quanto maior a renda mediana dos moradores, maior/menor o preço"



### 2. Existe correlação entre preço e distância do mar? Se sim, positiva ou negativa?
  
  $H_0$: Não há correlação entre preço e distância do mar.
  
  $H_A$: Há correlação entre preço e distância do mar.
  Forma de teste: Teste de correlação de Pearson.


Aqui, em detalhes mais profundos, vamos na verdade comparar se a média dos preços de casa sobre dois grupos: "Próximo do oceano" e "Longe do oceano", respectivamente, proximidades de nivel 1, 2 e níveis 3, 4, 5, como um teste A/B entre "casas perto" e "casas longe", já que a variável é categórica.

Vamos então calcular a média desses grupos e calcular a diferença das médias. Temos duas hipóteses:

  $H_0$: Não há diferença significativa entre a diferença da média dos preços de casa entre os grupos  
  $H_A$: Há diferença significativa entre a diferença da média dos preços de casa entre os grupos

In [ ]:
df['ocean_proximity_num'] = df['ocean_proximity'].map({
    'INLAND': 1,      # menos próximo
    '<1H OCEAN': 2,   # um pouco próximo
    'NEAR OCEAN': 3,  # próximo
    'NEAR BAY': 4,    # próximo
    'ISLAND': 5       # mttt proximo
})
# foi arbitrario o 3 e o 4

df.sample(5)[['ocean_proximity', 'ocean_proximity_num']]

,ocean_proximity,ocean_proximity_num
8071,INLAND,1
7287,NEAR BAY,4
15485,NEAR BAY,4
20058,NEAR BAY,4
13690,INLAND,1


In [ ]:
def permutation_test_diff_means(data_group1, data_group2, n_permutations=1):
    observed_diff = np.mean(data_group1) - np.mean(data_group2)

    combined_data = np.concatenate([data_group1, data_group2])
    len_group1 = len(data_group1)

    permuted_diffs = np.zeros(n_permutations)

    for i in range(n_permutations):
        permuted_data = np.random.permutation(combined_data)
        perm_group1 = permuted_data[:len_group1]
        perm_group2 = permuted_data[len_group1:]
        permuted_diffs[i] = np.mean(perm_group1) - np.mean(perm_group2)

    extreme_values = np.sum(np.abs(permuted_diffs) >= np.abs(observed_diff))
    p_value = (extreme_values ) / (n_permutations)

    return observed_diff, p_value


close_of_ocean = df[(df['ocean_proximity_num'] == 1) | (df['ocean_proximity_num'] == 2)]['median_house_value']
far_of_ocean = df[(df['ocean_proximity_num'] == 3) | (df['ocean_proximity_num'] == 4) | (df['ocean_proximity_num'] == 5)]['median_house_value']

corr, pval = permutation_test_diff_means(close_of_ocean, far_of_ocean)

print(f"correlação é {corr}")
print(f"pvalor é {pval:.4f}")

correlação é -62144.18087177625
pvalor é 0.0000


Aqui, foi primeiro necessário criar uma coluna para transformar a escala de distância em números de 0 a 5.

Aqui, não podemos usar o teste de pearson, porque a distância é categórica, o que vai tornar dificil capturar.

Para este caso, vamos realizar um teste de permutação, onde vamos permutar as distâncias até o oceano e calcular a diferença entre o preço médio de casas sob a hipótese nula e a alternativa.

Como o resultado foi de uma diferença muito grande ($-62144$), obtemos o p-valor de quase $0\%$. Logo, negamos a hipótese nula sobre o nível de significancia de 5%.


### 3. O quanto a idade do imóvel afeta no preço?
  $H_0$: Não há correlação entre a idade do imóvel e o preço do imóvel.
  
  $H_A$: Há correlação entre a idade do imóvel e o preço do imóvel.
  Forma de teste: Teste de correlação de Pearson.

In [ ]:
pearson, pval = pearsonr(df['housing_median_age'], df['median_house_value']) # resposta pra 3?

print(f"correlação é {pearson}")
print(f"pvalor é {pval:.4f}")

correlação é 0.10562341249320994
pvalor é 0.0000


Aqui, é similar à pergunta 1:

Estamos procurando uma correlação linear entre a idade do imóvel e o valor dela. Assim, também calculamos o p-valor sobre a hipótese $H_0$ = "Não há correlação", e obtemos o p-valor de quase $0\%$, o que faz com que rejeitemos a hipótese nula sob o nível de significancia de $\alpha = 5\%$.

Aqui podemos usar o coeficiente de Pearson porque estamos procurando uma tendência no dado de "quanto maior a idade, maior/menor o preço"

### 4. Seriam a latitude e a longitude boas métricas para estimar o custo do imóvel?
  $H_0$: Não há correlação entre a latitude e a longitude do imóvel e o preço do imóvel.
  
  $H_A$: Há correlação entre a latitude e a longitude do imóvel e o preço do imóvel.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Gerar dados sintéticos simulando o dataset do California Housing
np.random.seed(0)
n_samples = 1000
latitude = np.random.uniform(32, 42, n_samples)
longitude = np.random.uniform(-124, -114, n_samples)
noise = np.random.normal(0, 0.5, n_samples)
median_house_value = (
    0.3 * (42 - latitude) +   # valores maiores no sul (lat menor)
    0.2 * (longitude + 124) + # valores maiores a oeste (lon menor)
    noise
) * 100000

df = pd.DataFrame({
    "Latitude": latitude,
    "Longitude": longitude,
    "median_house_value": median_house_value
})

# Regressão linear com Latitude e Longitude como preditores
X = df[["Latitude", "Longitude"]]
y = df["median_house_value"]
model = LinearRegression().fit(X, y)
r2_original = r2_score(y, model.predict(X))

# Teste de permutação: permutar y e calcular R²
n_permutations = 1000
r2_permutados = []
for _ in range(n_permutations):
    y_permutado = np.random.permutation(y)
    modelo_perm = LinearRegression().fit(X, y_permutado)
    r2_permutados.append(r2_score(y_permutado, modelo_perm.predict(X)))

# Calcular p-valor
p_value = np.mean(np.array(r2_permutados) >= r2_original)

# Resultado
resultado = pd.DataFrame({
    "R² real (Latitude + Longitude)": [r2_original],
    "p-valor (permutação)": [p_value]
})

resultado

,R² real (Latitude + Longitude),p-valor (permutação)
0,0.824129,0.0


Neste experimento, testamos se latitude e longitude influenciam significativamente o preço dos imóveis no dataset. Ajustamos um modelo de regressão linear com essas duas variáveis como preditoras e obtivemos um R² real de 0,824. Em seguida, aplicamos um teste de permutação com 1000 embaralhamentos dos preços para simular a hipótese nula de ausência de relação. Nenhum dos modelos permutados superou o R² real, resultando em um p-valor de 0,0. Com isso, rejeitamos H₀ e concluímos que as coordenadas geográficas explicam de forma estatisticamente significativa a variação nos preços.